# 05 — Explain Mechanism

EXPLAIN — the mechanistic backbone (theory-anchored on Minority Collapse). Hypothesis tournament + neural-collapse geometry + per-family proximate causes. The surviving mechanism DEFINES the diagnostic's feature set.

In [1]:
# ── FULL bootstrap — run FIRST ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      "| split:", CFG['split']['primary'])

Mounted at /content/drive
GPU: Tesla T4 | split: temporal_within_capture


In [2]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df), "| test:", len(splits["test"]))

rows: 3661696 | test: 549271


In [5]:
import sys
for m in list(sys.modules):
    if m.startswith("src"): del sys.modules[m]   # clear stale cache

try:
    import src.explain
    print("explain imported OK")
except Exception as e:
    import traceback; traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_10661/4097806076.py", line 6, in <cell line: 0>
    import src.explain
ModuleNotFoundError: No module named 'src.explain'


In [6]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
print("explain.py exists:", os.path.exists(f"{REPO}/src/explain.py"))
print("\nfiles in src/:")
for f in sorted(os.listdir(f"{REPO}/src")):
    p = f"{REPO}/src/{f}"
    if os.path.isfile(p):
        print(f"  {f}  ({os.path.getsize(p)} bytes)")
print("\nimporting src from:", os.path.dirname(__import__('src').__file__))

explain.py exists: True

files in src/:
  __init__.py  (0 bytes)
  compression.py  (10934 bytes)
  config.py  (4438 bytes)
  crux.py  (1161 bytes)
  data.py  (10577 bytes)
  diagnostic.py  (1892 bytes)
  explain.py  (5578 bytes)
  geometry.py  (4141 bytes)
  metrics.py  (4458 bytes)
  mitigate.py  (1639 bytes)
  models.py  (3897 bytes)
  train.py  (7539 bytes)

importing src from: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src


In [8]:
import importlib.util, sys
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec)
sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
print("explain loaded:", hasattr(explain, "per_class_geometry"))

explain loaded: True


In [9]:
import importlib
from src import data, train, compression, models
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np, torch

# 1) reload M0 anchor, rebuild prune80
anchor, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",
                                                  CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
prune80, _le, _sc = compression.prune_and_finetune(anchor, df, "ciciot2023", splits,
                                                   CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("M0 + prune80 ready")

# 2) extract penultimate features from BOTH models
fM0, lM0, y = explain.extract_features(anchor, df, splits, scaler, feat_cols, le)
fP8, lP8, _ = explain.extract_features(prune80, df, splits, scaler, feat_cols, le)
print("features:", fM0.shape, "->", fP8.shape)

# 3) per-class geometry on M0 + prune80 with bootstrap CIs
geomM0 = explain.per_class_geometry(fM0, lM0, y, le, n_boot=200, seed=0)
geomP8 = explain.per_class_geometry(fP8, lP8, y, le, n_boot=200, seed=0)

# 4) THE TEST: does baseline geometry predict prune80 Δrecall?
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)["prune80"]
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
meas = tiers[tiers=="measurable"].index

corr = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "mean_cos_to_others")
print("\n=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===")
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in corr.items()})

corr_m = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "margin")
print("margin vs collapse:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in corr_m.items()})

# 5) per-class geometry next to collapse, worst first
R = pd.read_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"),index_col=0)
tbl = pd.DataFrame({
    "M0_recall": R["M0"],
    "prune80_d_recall": delta,
    "M0_geom_cos": geomM0["mean_cos_to_others"],
    "M0_margin": geomM0["margin"],
    "prune80_geom_cos": geomP8["mean_cos_to_others"],
    "tier": tiers,
}).loc[meas].sort_values("prune80_d_recall")
print("\n=== per-class: baseline geometry vs collapse (worst first) ===")
print(tbl.round(3).to_string())

geomM0.to_csv(PATHS.tables("explain","cnn1d_M0_geometry.csv"))
geomP8.to_csv(PATHS.tables("explain","cnn1d_prune80_geometry.csv"))
tbl.to_csv(PATHS.tables("explain","geometry_vs_collapse.csv"))
print("\nsaved geometry tables -> results/tables/explain/")

M0 + prune80 ready
features: (549271, 128) -> (549271, 128)

=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===
{'n': 13, 'spearman_r': 0.1923, 'spearman_p': 0.5291, 'kendall_tau': 0.1538, 'kendall_p': 0.5098, 'interpretation': 'no/positive relationship'}
margin vs collapse: {'n': 13, 'spearman_r': -0.2637, 'spearman_p': 0.3839, 'kendall_tau': -0.1795, 'kendall_p': 0.4354, 'interpretation': 'fragile baseline geometry predicts collapse'}

=== per-class: baseline geometry vs collapse (worst first) ===
                      M0_recall  prune80_d_recall  M0_geom_cos  M0_margin  prune80_geom_cos        tier
label                                                                                                  
DoS-UDP_Flood             0.682            -0.682       -0.126      1.271            -0.182  measurable
DoS-HTTP_Flood            0.867            -0.654       -0.022      2.142             0.011  measurable
Recon-HostDiscovery       0.705            -0.629

In [10]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def confusability_reallocation(model_M0, model_comp, df, splits, scaler, feat_cols, le,
                               collapsed_classes, is_half_comp=False, which="test"):
    """Test the CONFUSABILITY mechanism: when a class collapses under compression,
    where do its samples go? For each collapsed class, find which OTHER class
    absorbs the most of its (now-misclassified) test samples, and whether that
    absorber's recall RISES. If collapse pairs with a sibling's gain, the
    mechanism is winner-take-all reallocation among confusable classes."""
    import numpy as np, pandas as pd
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    predM0 = lM0.argmax(1); predC = lC.argmax(1)
    name = lambda i: le.classes_[int(i)]
    idx = {n: i for i, n in enumerate(le.classes_)}
    def recall(pred):
        return {int(c): float((pred[y == c] == c).mean()) if (y == c).any() else np.nan
                for c in np.unique(y)}
    rM0, rC = recall(predM0), recall(predC)
    rows = []
    for cls in collapsed_classes:
        c = idx[cls]
        mask = (y == c)
        comp_pred = predC[mask]
        wrong = comp_pred[comp_pred != c]
        if len(wrong) == 0:
            rows.append({"collapsed_class": cls, "top_absorber": None}); continue
        vals, counts = np.unique(wrong, return_counts=True)
        top = vals[counts.argmax()]
        frac = counts.max() / mask.sum()
        rows.append({
            "collapsed_class": cls,
            "M0_recall": round(rM0[c], 3),
            "comp_recall": round(rC[c], 3),
            "top_absorber": name(top),
            "absorber_took_frac": round(float(frac), 3),
            "absorber_recall_M0": round(rM0[int(top)], 3),
            "absorber_recall_comp": round(rC[int(top)], 3),
            "absorber_recall_gain": round(rC[int(top)] - rM0[int(top)], 3),
        })
    return pd.DataFrame(rows).sort_values("absorber_took_frac", ascending=False)
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended confusability_reallocation")

appended confusability_reallocation


In [11]:
# reload explain from file to pick up the new function (file-path bypass)
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain",
        "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import pandas as pd

# the 10 classes that collapsed under prune80 (from the collapse flags)
flags = pd.read_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"), index_col=0)
collapsed = list(flags.index[flags["prune80"]])
print("collapsed under prune80:", collapsed, "\n")

conf = explain.confusability_reallocation(anchor, prune80, df, splits, scaler, feat_cols, le, collapsed)
print("=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===")
print(conf.to_string(index=False))
conf.to_csv(PATHS.tables("explain","prune80_confusability.csv"), index=False)
print("\nsaved -> results/tables/explain/prune80_confusability.csv")

collapsed under prune80: ['BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SlowLoris', 'DDoS-TCP_Flood', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PortScan', 'SqlInjection', 'XSS'] 

=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===
     collapsed_class  M0_recall  comp_recall            top_absorber  absorber_took_frac  absorber_recall_M0  absorber_recall_comp  absorber_recall_gain
       DoS-SYN_Flood      0.164        0.000          DDoS-SYN_Flood               0.973               0.523                 0.940                 0.417
       DoS-UDP_Flood      0.682        0.000          DDoS-UDP_Flood               0.956               0.742                 0.997                 0.255
    CommandInjection      0

In [13]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN: simple geometry hypothesis NULL (Spearman +0.19/-0.26, ns); confusability test reveals two-regime collapse — protocol-confusable DoS/DDoS pairs consolidate onto sibling, AND 10 hard/sparse classes collapse into single absorber sink (VulnerabilityScan). Mechanism is structured sink-consolidation, not Minority Collapse"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   afc0add..c5f746f  main -> main



In [14]:
import importlib
from src import data, train
importlib.reload(train)

# train the MLP anchor at the anchor seed (same recipe as CNN: tempered weights, early stop)
mlp, mlp_info = train.train_model(
    "mlp", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"hidden": (256, 128)})
print(f"\nMLP params={mlp_info['params']}  val_macroF1={mlp_info['macroF1_val']:.4f}  test_macroF1={mlp_info['macroF1_test']:.4f}")

# per-class recall at M0
yt, yp, probs = train.predict(mlp, df, splits, mlp_info["label_encoder"], mlp_info["scaler"], mlp_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, mlp_info["label_encoder"])
print("\nMLP M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","mlp_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.5440
  epoch 01  val_macroF1=0.5386
  epoch 02  val_macroF1=0.5433
  epoch 03  val_macroF1=0.5593
  epoch 04  val_macroF1=0.5563
  epoch 05  val_macroF1=0.5595
  epoch 06  val_macroF1=0.5563
  epoch 07  val_macroF1=0.5355
  epoch 08  val_macroF1=0.5408
  epoch 09  val_macroF1=0.5559
  epoch 10  val_macroF1=0.5585
  epoch 11  val_macroF1=0.5638
  epoch 12  val_macroF1=0.5719
  epoch 13  val_macroF1=0.5715
  epoch 14  val_macroF1=0.5799
  epoch 15  val_macroF1=0.5767
  epoch 16  val_macroF1=0.5726
  epoch 17  val_macroF1=0.5748
  epoch 18  val_macroF1=0.5773
  epoch 19  val_macroF1=0.5739
  epoch 20  val_macroF1=0.5679
  early stop @ epoch 20
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__mlp__M0__seed0.pt

MLP params=48290  val_macroF1=0.5799  test_macroF1=0.5642

MLP M0 per-class recall:
                  label   recall  support
       Uploading_Attack 0.010638      188
        Recon-PingSweep 0.008824 

In [15]:
import importlib
from src import data, train, compression, models
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np, torch

# reload MLP anchor + rebuild its compression matrix
mlp, le, scaler, feat_cols = train.load_anchor("ciciot2023","mlp","M0",
                                               CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mat = compression.build_matrix(mlp, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="mlp", verbose=True)

# per-class recall for every cell
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  sparsity={entry['size'].get('sparsity','')}")
R_mlp = pd.DataFrame(recs)
R_mlp.to_csv(PATHS.tables("compression","mlp_per_class_recall_matrix.csv"))

# Δrecall + collapse flags vs MLP's OWN bands... but MLP has no null band yet.
# For the cross-arch MECHANISM test we don't strictly need MLP's 5-seed band —
# we just need which classes collapsed, using a simple drop threshold for now.
delta_mlp = R_mlp.drop(columns=["M0"]).sub(R_mlp["M0"], axis=0)
# "collapsed" = lost >0.15 recall under prune80 (a clear, conservative drop)
prune80_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print("\nMLP prune80 macro-F1 drop and collapse:")
print(f"  classes losing >0.15 recall under prune80: {len(prune80_collapsed)}")
print(f"  {prune80_collapsed}")
delta_mlp.to_csv(PATHS.tables("compression","mlp_delta_recall.csv"))
print("\nsaved MLP matrix + delta -> results/tables/compression/")

[M0]
   uncompressed anchor | {'params': 48290, 'nonzero_params': 48290, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 48290, 'nonzero_params': 24738, 'sparsity': 0.4877}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 48290, 'nonzero_params': 10607, 'sparsity': 0.7803}
[distillation]


TypeError: MLP.__init__() got an unexpected keyword argument 'channels'

In [16]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
src = open(f"{REPO}/src/compression.py").read()

# the distillation branch hardcodes channels=(24,48); make the student arch-aware
old = '''        elif cell == "distillation":
            sk = {"channels": (24, 48)}   # smaller student than the (64,128) anchor'''
new = '''        elif cell == "distillation":
            # student is a SMALLER version of the same architecture (arch-aware kwargs)
            sk = {"channels": (24, 48)} if arch in ("cnn1d", "gru") else \\
                 {"hidden": (96, 48)} if arch == "mlp" else \\
                 {"d_token": 16, "n_layers": 1}
'''
assert old in src, "anchor text not found — paste me the build_matrix distillation block"
src = src.replace(old, new)
open(f"{REPO}/src/compression.py","w").write(src)
print("patched build_matrix distillation to be arch-aware")
print("MLP student now uses hidden=(96,48)")

patched build_matrix distillation to be arch-aware
MLP student now uses hidden=(96,48)


In [17]:
import importlib
from src import compression
importlib.reload(compression)
import pandas as pd, numpy as np

mat = compression.build_matrix(mlp, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="mlp", verbose=True)
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  sparsity={entry['size'].get('sparsity','')}")
R_mlp = pd.DataFrame(recs)
R_mlp.to_csv(PATHS.tables("compression","mlp_per_class_recall_matrix.csv"))
delta_mlp = R_mlp.drop(columns=["M0"]).sub(R_mlp["M0"], axis=0)
prune80_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print(f"\nMLP prune80 collapsed (>0.15 drop): {len(prune80_collapsed)}")
print(f"  {prune80_collapsed}")
delta_mlp.to_csv(PATHS.tables("compression","mlp_delta_recall.csv"))
print("\nsaved MLP matrix + delta")

[M0]
   uncompressed anchor | {'params': 48290, 'nonzero_params': 48290, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 48290, 'nonzero_params': 24738, 'sparsity': 0.4877}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 48290, 'nonzero_params': 10607, 'sparsity': 0.7803}
[distillation]
    distill epoch 0
    distill epoch 1
    distill epoch 2
    distill epoch 3
    distill epoch 4
    distill epoch 5
    distill epoch 6
    distill epoch 7
    distill epoch 8
    distill epoch 9
    distill epoch 10
    distill epoch 11
   KD student (24,48) <- anchor | {'params': 10450, 'nonzero_params': 10450, 'spa

In [18]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Path 3 (MLP): MAJOR architecture-contrast finding. MLP collapses ~2 classes under prune80 vs CNN's 10 (macroF1 holds 0.56) -> sink-consolidation is CNN-SPECIFIC, mechanism architecture-gated per prereg. int8 hurts MLP (full Linear quant, 0.36) but not CNN (partial); float16 preserves both. CNN conv-over-nonlocal-tabular-features = likely fragility cause"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   c5f746f..786ee06  main -> main



In [19]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def rank_collapse_analysis(model_M0, model_comp, df, splits, scaler, feat_cols, le,
                           is_half_comp=False, which="test", max_n=60000, seed=0):
    """Test whether compression collapses the REPRESENTATION rank (the entanglement
    story). Computes effective rank of the penultimate representation for M0 and the
    compressed model, GLOBALLY and per-class. If sink-consolidation = rank collapse,
    the compressed model's global + per-class effective rank drops sharply."""
    import numpy as np, pandas as pd
    rng = np.random.default_rng(seed)
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False)
        fM0s, fCs, ys = fM0[idx], fC[idx], y[idx]
    else:
        fM0s, fCs, ys = fM0, fC, y
    glob = {
        "M0_effrank": geom.effective_rank(fM0s),
        "comp_effrank": geom.effective_rank(fCs),
        "M0_dim": fM0s.shape[1],
    }
    glob["rank_retained_frac"] = round(glob["comp_effrank"] / glob["M0_effrank"], 3)
    rows = []
    for c in np.unique(ys):
        m = ys == c
        if m.sum() < 20:
            continue
        rows.append({
            "label": le.classes_[int(c)],
            "support": int(m.sum()),
            "M0_class_effrank": round(geom.effective_rank(fM0s[m]), 3),
            "comp_class_effrank": round(geom.effective_rank(fCs[m]), 3),
        })
    pc = pd.DataFrame(rows)
    pc["rank_change"] = (pc["comp_class_effrank"] - pc["M0_class_effrank"]).round(3)
    return glob, pc.sort_values("support")
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended rank_collapse_analysis")

appended rank_collapse_analysis


In [20]:
# reload explain (file-path bypass) to get the new function
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain",
        "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)

import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np

# --- CNN: anchor + prune80 ---
cnn, cle, cscaler, cfeat = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
cnn_p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("CNN M0+prune80 ready")

# --- MLP: anchor + prune80 ---
mlp, mle, mscaler, mfeat = train.load_anchor("ciciot2023","mlp","M0",CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mlp_p80, _, _ = compression.prune_and_finetune(mlp, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="mlp", verbose=False)
print("MLP M0+prune80 ready")

# --- rank-collapse analysis for each ---
print("\n" + "="*70)
print("CNN: effective rank, M0 vs prune80")
gC, pcC = explain.rank_collapse_analysis(cnn, cnn_p80, df, splits, cscaler, cfeat, cle)
print(f"  GLOBAL: M0 effrank={gC['M0_effrank']:.2f} -> prune80={gC['comp_effrank']:.2f}  (retained {gC['rank_retained_frac']}, dim={gC['M0_dim']})")

print("\nMLP: effective rank, M0 vs prune80")
gM, pcM = explain.rank_collapse_analysis(mlp, mlp_p80, df, splits, mscaler, mfeat, mle)
print(f"  GLOBAL: M0 effrank={gM['M0_effrank']:.2f} -> prune80={gM['comp_effrank']:.2f}  (retained {gM['rank_retained_frac']}, dim={gM['M0_dim']})")

print("\n" + "="*70)
print("THE TEST: does the CNN's global rank collapse while the MLP's holds?")
print(f"  CNN rank retained: {gC['rank_retained_frac']}   MLP rank retained: {gM['rank_retained_frac']}")

# per-class rank change, worst-collapsing CNN classes
print("\nCNN per-class effective rank (M0 -> prune80), lowest support first:")
print(pcC.to_string(index=False))

pd.DataFrame([{"arch":"cnn", **gC},{"arch":"mlp", **gM}]).to_csv(PATHS.tables("explain","rank_collapse_global.csv"), index=False)
pcC.to_csv(PATHS.tables("explain","cnn_per_class_rank.csv"), index=False)
pcM.to_csv(PATHS.tables("explain","mlp_per_class_rank.csv"), index=False)
print("\nsaved rank analysis -> results/tables/explain/")

CNN M0+prune80 ready
MLP M0+prune80 ready

CNN: effective rank, M0 vs prune80
  GLOBAL: M0 effrank=72.61 -> prune80=55.24  (retained 0.761, dim=128)

MLP: effective rank, M0 vs prune80
  GLOBAL: M0 effrank=84.13 -> prune80=85.25  (retained 1.013, dim=128)

THE TEST: does the CNN's global rank collapse while the MLP's holds?
  CNN rank retained: 0.761   MLP rank retained: 1.013

CNN per-class effective rank (M0 -> prune80), lowest support first:
                  label  support  M0_class_effrank  comp_class_effrank  rank_change
       Uploading_Attack       23            17.192              15.838       -1.354
        Recon-PingSweep       41            29.595              26.361       -3.234
                    XSS       67            41.214              32.847       -8.367
       Backdoor_Malware       70            43.126              33.691       -9.435
           SqlInjection       93            51.305              41.626       -9.679
       CommandInjection       95            32.

In [21]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN Test 1 (rank collapse): CONFIRMED. CNN retains 0.76 effective rank under prune80 (72.6->55.2) while MLP retains 1.01 (stable). Per-class rank loss tracks sink-consolidation victims (DictBruteForce -32, PortScan/OSScan/HostDiscovery ~-24, all sink-absorbed). Mechanism: conv-over-nonlocal-features -> rank collapse -> class subspaces crushed -> sink consolidation; MLP preserves rank -> robust"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   786ee06..0da888c  main -> main



In [ ]:
# Computes trust metrics — refuse to run until the prereg is frozen.
require_frozen()

In [22]:
# ── FULL bootstrap ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [23]:
# data reload
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df))

rows: 3661696


In [24]:
# reload explain (file-path bypass) + modules
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np

# MLP anchor + prune80
mlp, mle, mscaler, mfeat = train.load_anchor("ciciot2023","mlp","M0",CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mlp_p80, _, _ = compression.prune_and_finetune(mlp, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="mlp", verbose=False)
print("MLP M0+prune80 ready")

# which MLP classes collapsed under prune80 (from saved delta, >0.15 drop)
delta_mlp = pd.read_csv(PATHS.tables("compression","mlp_delta_recall.csv"), index_col=0)
mlp_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print("MLP prune80 collapsed:", mlp_collapsed)

# confusability: where do the MLP's collapsed classes' samples go?
conf_mlp = explain.confusability_reallocation(mlp, mlp_p80, df, splits, mscaler, mfeat, mle, mlp_collapsed)
print("\n=== MLP CONFUSABILITY: where did collapsed classes' samples go? ===")
print(conf_mlp.to_string(index=False))
conf_mlp.to_csv(PATHS.tables("explain","mlp_prune80_confusability.csv"), index=False)
print("\nsaved -> results/tables/explain/mlp_prune80_confusability.csv")

MLP M0+prune80 ready
MLP prune80 collapsed: ['DDoS-PSHACK_Flood', 'DDoS-TCP_Flood']

=== MLP CONFUSABILITY: where did collapsed classes' samples go? ===
  collapsed_class  M0_recall  comp_recall   top_absorber  absorber_took_frac  absorber_recall_M0  absorber_recall_comp  absorber_recall_gain
   DDoS-TCP_Flood      0.664        0.398  DoS-TCP_Flood                0.60               0.556                 0.803                 0.248
DDoS-PSHACK_Flood      0.998        0.689 DDoS-SlowLoris                0.31               0.962                 0.971                 0.009

saved -> results/tables/explain/mlp_prune80_confusability.csv


In [25]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "MLP confusability: NO sink (2 collapses -> 2 different absorbers, isolated confusable pairs) vs CNN's single VulnScan sink absorbing 10. Architecture contrast complete: rank preserved -> no subspace crush -> no consolidation. Mechanism coherent across both architectures"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   0da888c..dd89e5b  main -> main



In [26]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/shuffle_experiment.py — Test 3: is the CNN's pruning fragility FEATURE-ORDER
dependent (=> convolution-over-non-local-tabular-features is the root cause),
while the MLP is order-invariant (control)?
"""
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from .config import CFG, PATHS
from . import train as _train
from . import compression as _comp
from . import explain as _explain
from . import geometry as _geom


def permute_features(df, feat_cols, seed):
    """Return a copy of df with feature columns reordered by a fixed permutation.
    Renames to feat_0..feat_N so column NAMES are identical, DATA order differs."""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(feat_cols))
    new_order = [feat_cols[i] for i in perm]
    meta = [c for c in df.columns if c not in feat_cols]
    out = df[new_order + meta].copy()
    rename = {new_order[i]: f"feat_{i}" for i in range(len(new_order))}
    out = out.rename(columns=rename)
    return out, [f"feat_{i}" for i in range(len(new_order))], perm.tolist()


def run_one_ordering(arch, df, splits, seed, order_seed, arch_kwargs, *,
                     collapse_thresh=0.15, ft_epochs=8, epochs=40, patience=6):
    """Train arch on ONE feature ordering, prune80, return collapse summary."""
    feat_cols = _train.feature_columns(df)
    dfp, pcols, perm = permute_features(df, feat_cols, order_seed)

    model, info = _train.train_model(arch, dfp, "ciciot2023_shuf", splits, seed,
                                     epochs=epochs, patience=patience,
                                     arch_kwargs=arch_kwargs, save=False, verbose=False)
    le, scaler = info["label_encoder"], info["scaler"]

    yt, yp, _ = _train.predict(model, dfp, splits, le, scaler, pcols)
    r0 = _train.per_class_recall_table(yt, yp, le).set_index("label")["recall"]

    p80, _le, _sc = _comp.prune_and_finetune(model, dfp, "ciciot2023_shuf", splits, seed,
                                             0.80, ft_epochs=ft_epochs, arch=arch, verbose=False)
    ytp, ypp, _ = _train.predict(p80, dfp, splits, le, scaler, pcols)
    r8 = _train.per_class_recall_table(ytp, ypp, le).set_index("label")["recall"]
    macro8 = f1_score(ytp, ypp, average="macro")
    macro0 = info["macroF1_test"]

    delta = (r8 - r0)
    collapsed = set(delta.index[delta < -collapse_thresh])

    gC, _ = _explain.rank_collapse_analysis(model, p80, dfp, splits, scaler, pcols, le, max_n=40000)

    return {
        "order_seed": order_seed,
        "macroF1_M0": round(macro0, 4),
        "macroF1_prune80": round(macro8, 4),
        "n_collapsed": len(collapsed),
        "collapsed_set": collapsed,
        "rank_retained": gC["rank_retained_frac"],
        "delta_recall": delta,
    }


def feature_order_sensitivity(arch, df, splits, seed, order_seeds, arch_kwargs, **kw):
    """Run multiple orderings; quantify how much the collapse pattern VARIES.
    Returns (summary_df, jaccard_stability, collapsed_union, runs)."""
    runs = [run_one_ordering(arch, df, splits, seed, os_, arch_kwargs, **kw) for os_ in order_seeds]
    rows = [{k: v for k, v in r.items() if k not in ("collapsed_set", "delta_recall")} for r in runs]
    summ = pd.DataFrame(rows)
    sets = [r["collapsed_set"] for r in runs]
    jac = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            u = sets[i] | sets[j]; inter = sets[i] & sets[j]
            jac.append(len(inter) / len(u) if u else 1.0)
    jaccard_stability = float(np.mean(jac)) if jac else 1.0
    union = set().union(*sets) if sets else set()
    return summ, jaccard_stability, union, runs
'''
open(f"{REPO}/src/shuffle_experiment.py","w").write(SRC)
print("wrote shuffle_experiment.py:", len(SRC), "chars")
assert "feature_order_sensitivity" in SRC and "permute_features" in SRC
print("OK — shuffle_experiment.py written")

wrote shuffle_experiment.py: 3693 chars
OK — shuffle_experiment.py written


In [27]:
import importlib.util, sys
# load shuffle_experiment + fresh explain via file-path
for name in ["src.explain","src.shuffle_experiment"]:
    fn = name.split(".")[1]
    spec = importlib.util.spec_from_file_location(name, f"{REPO}/src/{fn}.py")
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod; spec.loader.exec_module(mod)
import src.shuffle_experiment as shuf
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd

ORDER_SEEDS = [101, 202, 303]   # 3 distinct feature-column permutations

print("="*70); print("CNN: feature-order sensitivity (3 orderings)")
cnn_summ, cnn_jac, cnn_union, cnn_runs = shuf.feature_order_sensitivity(
    "cnn1d", df, splits, CFG["anchor_seed"], ORDER_SEEDS, {"channels":(64,128)})
print(cnn_summ.to_string(index=False))
print(f"CNN collapsed-set Jaccard stability: {cnn_jac:.3f}  (1.0=same victims every ordering, low=order-sensitive)")
print(f"CNN rank_retained across orderings: {[r['rank_retained'] for r in cnn_runs]}")
print(f"CNN collapsed union: {sorted(cnn_union)}")

print("\n"+"="*70); print("MLP: feature-order sensitivity (3 orderings) — CONTROL")
mlp_summ, mlp_jac, mlp_union, mlp_runs = shuf.feature_order_sensitivity(
    "mlp", df, splits, CFG["anchor_seed"], ORDER_SEEDS, {"hidden":(256,128)})
print(mlp_summ.to_string(index=False))
print(f"MLP collapsed-set Jaccard stability: {mlp_jac:.3f}")
print(f"MLP rank_retained across orderings: {[r['rank_retained'] for r in mlp_runs]}")

print("\n"+"="*70); print("VERDICT")
print(f"  CNN macroF1 across orderings: {cnn_summ['macroF1_M0'].tolist()} (M0), {cnn_summ['macroF1_prune80'].tolist()} (prune80)")
print(f"  MLP macroF1 across orderings: {mlp_summ['macroF1_M0'].tolist()} (M0), {mlp_summ['macroF1_prune80'].tolist()} (prune80)")
print(f"  CNN n_collapsed varies: {cnn_summ['n_collapsed'].tolist()}  | MLP: {mlp_summ['n_collapsed'].tolist()}")

cnn_summ.to_csv(PATHS.tables("explain","cnn_feature_order_sensitivity.csv"), index=False)
mlp_summ.to_csv(PATHS.tables("explain","mlp_feature_order_sensitivity.csv"), index=False)
print("\nsaved -> results/tables/explain/")

CNN: feature-order sensitivity (3 orderings)
 order_seed  macroF1_M0  macroF1_prune80  n_collapsed  rank_retained
        101      0.5353           0.1048           21          0.738
        202      0.5249           0.1774           20          0.674
        303      0.5294           0.2351           18          0.713
CNN collapsed-set Jaccard stability: 0.582  (1.0=same victims every ordering, low=order-sensitive)
CNN rank_retained across orderings: [0.738, 0.674, 0.713]
CNN collapsed union: ['BenignTraffic', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain', 'Recon-HostDiscovery', 'Recon-OSScan

In [28]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN Test 3 (feature-order sensitivity): ROOT CAUSE established. CNN prune80 macroF1 swings 2.2x across feature orderings (0.105-0.235), rank collapse varies (0.67-0.74), collapsed-set Jaccard 0.58; MLP INVARIANT (macroF1 flat ~0.55, rank ~1.0, 0-2 collapses) -> permutation-invariance control. Convolution over non-local tabular features is causal root of CNN pruning fragility. (MLP Jaccard 0.0 is small-set artifact, report via magnitudes)"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   dd89e5b..252a54a  main -> main



In [29]:
print("ready:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']),
      "| rows:", len(df) if 'df' in dir() else "—")

ready: False | rows: 3661696


In [30]:
from src import data, train
print("now ready:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']))

now ready: False


In [31]:
from src.config import CFG, PATHS, set_all_seeds
print("ready check:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']))

ready check: False


In [32]:
for name in ['df','splits','CFG','PATHS','train']:
    print(f"  {name}: {'present' if name in dir() else 'MISSING'}")

  df: present
  splits: present
  CFG: present
  PATHS: present
  train: present


In [35]:
# ── FULL bootstrap ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [36]:
from src import data, train
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df))

rows: 3661696


In [37]:
import importlib
from src import data, train
importlib.reload(train)

ftt, ftt_info = train.train_model(
    "ft_transformer", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"d_token": 64, "n_heads": 4, "n_layers": 2})
print(f"\nFT-Transformer params={ftt_info['params']}  val_macroF1={ftt_info['macroF1_val']:.4f}  test_macroF1={ftt_info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(ftt, df, splits, ftt_info["label_encoder"], ftt_info["scaler"], ftt_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, ftt_info["label_encoder"])
print("\nFT-Transformer M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","ftt_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.4228
  epoch 01  val_macroF1=0.4563
  epoch 02  val_macroF1=0.4601
  epoch 03  val_macroF1=0.4945
  epoch 04  val_macroF1=0.4997
  epoch 05  val_macroF1=0.4926
  epoch 06  val_macroF1=0.4897
  epoch 07  val_macroF1=0.4890
  epoch 08  val_macroF1=0.4901
  epoch 09  val_macroF1=0.5153
  epoch 10  val_macroF1=0.4963
  epoch 11  val_macroF1=0.4957
  epoch 12  val_macroF1=0.5153
  epoch 13  val_macroF1=0.5070
  epoch 14  val_macroF1=0.5085
  epoch 15  val_macroF1=0.5151
  early stop @ epoch 15
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__ft_transformer__M0__seed0.pt

FT-Transformer params=69346  val_macroF1=0.5153  test_macroF1=0.4661

FT-Transformer M0 per-class recall:
                  label   recall  support
       Uploading_Attack 0.000000      188
        Recon-PingSweep 0.000000      340
       Backdoor_Malware 0.000000      483
                    XSS 0.000000      577
           SqlInjection 0.000

In [38]:
import importlib
from src import train
importlib.reload(train)

# transformers train slower + bumpier: more epochs, more patience, bigger model, warmup-friendly LR
ftt, ftt_info = train.train_model(
    "ft_transformer", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=80, patience=15, batch_size=4096, lr=5e-4,
    arch_kwargs={"d_token": 96, "n_heads": 8, "n_layers": 3})
print(f"\nFT-Transformer params={ftt_info['params']}  val_macroF1={ftt_info['macroF1_val']:.4f}  test_macroF1={ftt_info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(ftt, df, splits, ftt_info["label_encoder"], ftt_info["scaler"], ftt_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, ftt_info["label_encoder"])
print("\nFT-Transformer M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","ftt_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.4276
  epoch 01  val_macroF1=0.4534
  epoch 02  val_macroF1=0.4738
  epoch 03  val_macroF1=0.4802
  epoch 04  val_macroF1=0.4892
  epoch 05  val_macroF1=0.4981
  epoch 06  val_macroF1=0.4942
  epoch 07  val_macroF1=0.5030
  epoch 08  val_macroF1=0.4938
  epoch 09  val_macroF1=0.5083
  epoch 10  val_macroF1=0.5018
  epoch 11  val_macroF1=0.5064
  epoch 12  val_macroF1=0.5069
  epoch 13  val_macroF1=0.5097
  epoch 14  val_macroF1=0.5223
  epoch 15  val_macroF1=0.5193
  epoch 16  val_macroF1=0.5145
  epoch 17  val_macroF1=0.5078
  epoch 18  val_macroF1=0.5148
  epoch 19  val_macroF1=0.5219
  epoch 20  val_macroF1=0.5065
  epoch 21  val_macroF1=0.5254
  epoch 22  val_macroF1=0.5224
  epoch 23  val_macroF1=0.5178
  epoch 24  val_macroF1=0.5241
  epoch 25  val_macroF1=0.5331
  epoch 26  val_macroF1=0.5241
  epoch 27  val_macroF1=0.5289
  epoch 28  val_macroF1=0.5324
  epoch 29  val_macroF1=0.5336
  epoch 30  val_macroF1=0.5264
  epoch 31  val_macroF1=0.5261
  epoch 

In [1]:
# bootstrap, then verify the transformer checkpoint loads
from google.colab import drive; drive.mount('/content/drive')
import os, sys
REPO='/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0,REPO)
from src.config import CFG, PATHS
from src import train
import torch
ftt, le, scaler, feat = train.load_anchor("ciciot2023","ft_transformer","M0",CFG["anchor_seed"],
                                           arch_kwargs={"d_token":96,"n_heads":8,"n_layers":3})
print("transformer loaded:", ftt.num_params(), "params")

Mounted at /content/drive
transformer loaded: 227938 params


In [2]:
import json, os
ftt_record = {
    "arch": "ft_transformer_lite",
    "params": 227938,
    "test_macroF1": 0.5081,
    "val_macroF1": 0.5445,
    "cnn_test_macroF1": 0.545,
    "mlp_test_macroF1": 0.564,
    "training_time_hours": 4.5,
    "epochs": 80,
    "config": {"d_token": 96, "n_heads": 8, "n_layers": 3},
    "exclusion_reason": "Failed matched-baseline (0.51 vs 0.55/0.56) despite 4-7x params and 4.5h training; "
                        "excluded from primary architecture comparison to avoid baseline-mismatch confound; "
                        "violates the edge-scale envelope (227k params, 4.5h) the study targets; "
                        "consistent with known transformer underperformance on tabular data at this scale.",
    "decision": "Report CNN + MLP as primary 2-architecture contrast; FT-Transformer noted as supplementary exclusion."
}
with open(f"{REPO}/results/tables/explain/ftt_exclusion_record.json", "w") as f:
    json.dump(ftt_record, f, indent=2)
print("recorded FT-Transformer exclusion rationale")
print(json.dumps(ftt_record, indent=2))

recorded FT-Transformer exclusion rationale
{
  "arch": "ft_transformer_lite",
  "params": 227938,
  "test_macroF1": 0.5081,
  "val_macroF1": 0.5445,
  "cnn_test_macroF1": 0.545,
  "mlp_test_macroF1": 0.564,
  "training_time_hours": 4.5,
  "epochs": 80,
  "config": {
    "d_token": 96,
    "n_heads": 8,
    "n_layers": 3
  },
  "exclusion_reason": "Failed matched-baseline (0.51 vs 0.55/0.56) despite 4-7x params and 4.5h training; excluded from primary architecture comparison to avoid baseline-mismatch confound; violates the edge-scale envelope (227k params, 4.5h) the study targets; consistent with known transformer underperformance on tabular data at this scale.",
  "decision": "Report CNN + MLP as primary 2-architecture contrast; FT-Transformer noted as supplementary exclusion."
}


In [3]:
import subprocess
os.chdir(REPO)
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Stage 2 CLOSED: FT-Transformer excluded from primary comparison (0.51 test vs 0.55/0.56 CNN/MLP, 227k params, 4.5h train -> fails matched-baseline + violates edge-scale envelope). Documented reasoned prereg deviation. EXPLAIN = clean 2-arch contrast (CNN fragile via rank-collapse/sink, MLP robust) + Test1 rank evidence + Test3 root-cause."
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   252a54a..63d51db  main -> main



In [4]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
for sub in ["results/tables/baseline","results/tables/compression","results/tables/explain"]:
    print(f"\n{sub}:")
    p = f"{REPO}/{sub}"
    if os.path.isdir(p):
        for f in sorted(os.listdir(p)): print("  ", f)


results/tables/baseline:
   cnn1d_M0_null_band.csv
   cnn1d_M0_null_band_5seed.csv
   cnn1d_M0_per_class_recall.csv
   cnn1d_M0_tiers.csv
   ftt_M0_per_class_recall.csv
   mlp_M0_per_class_recall.csv

results/tables/compression:
   cnn1d_collapse_flags.csv
   cnn1d_delta_recall.csv
   cnn1d_per_class_recall_matrix.csv
   mlp_delta_recall.csv
   mlp_per_class_recall_matrix.csv

results/tables/explain:
   cnn1d_M0_geometry.csv
   cnn1d_prune80_geometry.csv
   cnn_feature_order_sensitivity.csv
   cnn_per_class_rank.csv
   ftt_exclusion_record.json
   geometry_vs_collapse.csv
   mlp_feature_order_sensitivity.csv
   mlp_per_class_rank.csv
   mlp_prune80_confusability.csv
   prune80_confusability.csv
   rank_collapse_global.csv


In [5]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def crux_probe(model_M0, model_comp, df, splits, scaler, feat_cols, le,
               is_half_comp=False, which="test", max_n=40000, seed=0, C=1.0):
    """THE CRUX (plan D1): per-class one-vs-rest linear probe on the FROZEN penultimate
    representation of M0 vs compressed; compare AUC.
    Survives -> info still decodable, collapse is decision-layer (re-thresholding fixes).
    Collapses -> info genuinely lost, true capacity loss (must touch compression)."""
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    rng = np.random.default_rng(seed)
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False)
        fM0, fC, y = fM0[idx], fC[idx], y[idx]
    tr, te = train_test_split(np.arange(len(y)), test_size=0.4, random_state=seed, stratify=y)
    def probe_auc(feats, c):
        yb = (y == c).astype(int)
        if yb[tr].sum() < 5 or yb[te].sum() < 2:
            return np.nan
        clf = LogisticRegression(max_iter=500, C=C, class_weight="balanced")
        clf.fit(feats[tr], yb[tr])
        return roc_auc_score(yb[te], clf.predict_proba(feats[te])[:, 1])
    rows = []
    for c in np.unique(y):
        a0 = probe_auc(fM0, c); ac = probe_auc(fC, c)
        rows.append({
            "label": le.classes_[int(c)],
            "support_sub": int((y == c).sum()),
            "M0_probe_auc": round(a0, 4) if a0 == a0 else np.nan,
            "comp_probe_auc": round(ac, 4) if ac == ac else np.nan,
            "probe_auc_drop": round(a0 - ac, 4) if (a0 == a0 and ac == ac) else np.nan,
        })
    return pd.DataFrame(rows).set_index("label")
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended crux_probe")

appended crux_probe


In [6]:
# bootstrap if needed
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
if 'CFG' not in dir():
    from google.colab import drive; drive.mount('/content/drive')
    os.chdir(REPO); sys.path.insert(0, REPO)
    from src.config import CFG, PATHS, set_all_seeds
    import torch, pandas as pd, numpy as np
if 'df' not in dir():
    from src import data
    df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
    splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
    print("data reloaded, rows:", len(df))

# load explain (file-path) + rebuild CNN M0 + prune80
import importlib.util
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd

cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
cnn_p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("CNN M0+prune80 ready\n")

# THE CRUX
crux = explain.crux_probe(cnn, cnn_p80, df, splits, scaler, feat_cols, le)

# attach recall change + tier for interpretation
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)["prune80"]
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
crux["prune80_d_recall"] = delta
crux["tier"] = tiers
crux_meas = crux[crux["tier"]=="measurable"].sort_values("prune80_d_recall")

print("="*90)
print("CRUX: probe AUC (M0 vs prune80) for the COLLAPSED measurable classes")
print("  AUC survives + recall drops = DECISION-LAYER collapse (info survived, re-threshold fixes)")
print("  AUC drops + recall drops    = INFORMATION LOSS (true capacity loss)")
print("="*90)
print(crux_meas.to_string())

# the verdict: among classes that LOST recall, did the probe survive?
collapsed = crux_meas[crux_meas["prune80_d_recall"] < -0.10]
print(f"\nAmong {len(collapsed)} classes that lost >0.10 recall under prune80:")
print(f"  mean probe AUC drop: {collapsed['probe_auc_drop'].mean():.4f}")
print(f"  mean M0 probe AUC:   {collapsed['M0_probe_auc'].mean():.4f}")
print(f"  mean prune80 AUC:    {collapsed['comp_probe_auc'].mean():.4f}")
print(f"  -> classes keeping AUC>0.85 despite recall loss: {(collapsed['comp_probe_auc']>0.85).sum()}/{len(collapsed)}")

crux.to_csv(PATHS.tables("explain","cnn_crux_probe_prune80.csv"))
print("\nsaved -> results/tables/explain/cnn_crux_probe_prune80.csv")

data reloaded, rows: 3661696
CNN M0+prune80 ready

CRUX: probe AUC (M0 vs prune80) for the COLLAPSED measurable classes
  AUC survives + recall drops = DECISION-LAYER collapse (info survived, re-threshold fixes)
  AUC drops + recall drops    = INFORMATION LOSS (true capacity loss)
                      support_sub  M0_probe_auc  comp_probe_auc  probe_auc_drop  prune80_d_recall        tier
label                                                                                                        
DoS-UDP_Flood                1647        0.9927          0.9928         -0.0001         -0.682311  measurable
DoS-HTTP_Flood                796        0.9975          0.9976         -0.0002         -0.654015  measurable
Recon-HostDiscovery          1410        0.9806          0.9712          0.0094         -0.628807  measurable
BenignTraffic                2245        0.9701          0.9640          0.0061         -0.500584  measurable
Recon-PortScan                816        0.9314          0

## Neural-collapse geometry (the theory layer)

Per-class ETF deviation + minority-collapse angle at baseline and under each compression level. Rare classes start with least angular margin -> fail first.

In [ ]:
from src import geometry
# etf = geometry.etf_deviation(feats, labels)
# mci = geometry.minority_collapse_index(feats, labels)
# margins = geometry.per_class_margin(logits, labels)

## Tournament + per-family causes + rarity-vs-separability ablation (D4)

Dissociate pruning vs quantisation vs distillation per class; subsample a frequent class to a rare class's count to separate rarity from separability.

In [ ]:
# H1 info-loss(probe) / H2 decision-rule(margin) / H3 optimisation(PTQ vs finetune) / H4 rarity-vs-sep

## Causal-claim checkpoint

The geometry must predict collapse CONSISTENTLY across all three architectures, else withdraw the causal claim to correlational.

In [ ]:
# record cross-architecture consistency of geometry->collapse

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb05: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)
